In [ ]:
"""
Multi-Dimensional Scaling Analysis
Combines binary categorical data (platforms, modality, task, models)
with topic distributions to map terms and topics in 2D space based on co-occurrence
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import MDS
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import euclidean_distances
from scipy.cluster.hierarchy import linkage, dendrogram
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
color_cats = {
    'platform': "Blues",
    'modality': "Greens",
    'model': "Oranges",
    'task': "Reds",
    }


In [ ]:
def load_data(binary_csv_path1, binary_csv_path2, topics_csv_path, meta_csv_path, binary_topics = True):
    """
    Load binary categories and topic distributions

    Parameters:
    -----------
    binary_csv_path : str
        Path to CSV with binary categories (platforms, modality, task, models)
    topics_csv_path : str
        Path to CSV with topic distributions (9 topics per paper)

    Returns:
    --------
    binary_df : DataFrame
        Binary categories data
    topics_df : DataFrame
        Topic distributions data
    """
    print("Loading data...")

    # Load binary categories
    binary_df1 = pd.read_csv(binary_csv_path1)
    binary_cols1 = [col for col in binary_df1.columns if '_' in col and any(col.startswith(prefix)  for prefix in ['platform_'])]+ ['filename']
    binary_df1 = binary_df1[binary_cols1]

    binary_df2 = pd.read_csv(binary_csv_path2)
    binary_cols2 = [col for col in binary_df2.columns if '_' in col and any(col.startswith(prefix)  for prefix in ['task_', 'model_', 'modality_'])][4:]+ ['filename']
    binary_df2 = binary_df2[binary_cols2]

    binary_df = binary_df1.merge(binary_df2, on='filename', how='inner').set_index('filename').reset_index()
    print(f"Binary categories shape: {binary_df.shape}")
    print(f"Binary categories columns: {list(binary_df.columns)}")

    # Load meta data
    df_meta = pd.read_csv(meta_csv_path)
    df_meta['filename'] = df_meta['File_Name'] + '.txt'
    df_prep = binary_df2.merge(df_meta[['Key', 'filename']], left_on='filename', right_on='filename', how='inner')

    # Load topic distributions
    topics_df = pd.read_csv(topics_csv_path, encoding="utf-8", sep=';', decimal=",")
    if not binary_topics:
        topic_cols = [col for col in topics_df.columns if '_' in col and any(col.startswith(prefix) for prefix in ['Prob_Topic_'])] + ['Key', 'Assigned_Topic', 'Topic_Label']

    else:
        for i in sorted(topics_df['Assigned_Topic'].unique()):
            topics_df['Binary_Topic_{}'.format(i)] = topics_df['Assigned_Topic'].apply(lambda x: x==i).astype(int)
        topic_cols = [col for col in topics_df.columns if '_' in col and any(col.startswith(prefix) for prefix in ['Binary_Topic_'])] + ['Key', 'Assigned_Topic', 'Topic_Label']

    topics_df = topics_df[topic_cols]
    topics_df = df_prep.merge(topics_df, on='Key', how='inner')[topic_cols + ['filename']].set_index('filename').reset_index()
    print(f"Topic distributions shape: {topics_df.shape}")
    print(f"Topic columns: {list(topics_df.columns)}")

    return binary_df, topics_df

In [ ]:
def combine_datasets(binary_df, topics_df, paper_id_col='filename'):
    """
    Combine binary categories and topic distributions

    Parameters:
    -----------
    binary_df : DataFrame
        Binary categories
    topics_df : DataFrame
        Topic distributions
    paper_id_col : str
        Column name for paper identifier (default: 'paper_id')

    Returns:
    --------
    combined_df : DataFrame
        Combined dataset
    """
    print("\nCombining datasets...")

    # If paper_id column exists, merge on it; otherwise merge by index
    if paper_id_col in binary_df.columns and paper_id_col in topics_df.columns:
        combined_df = pd.merge(binary_df, topics_df.iloc[:, :-3], on=paper_id_col, how='inner')
        print(f"Merged on '{paper_id_col}' column")
    else:
        # Reset index and merge
        binary_df = binary_df.reset_index(drop=True)
        topics_df = topics_df.reset_index(drop=True)
        combined_df = pd.concat([binary_df, topics_df], axis=1)
        print("Concatenated by index")

    print(f"Combined shape: {combined_df.shape}")

    return combined_df

In [ ]:
def get_distance_matrix(df, feature_cols, distance_method):
    print(f"\nCalculating distances using: {distance_method}")
    # Extract feature matrix
    feature_matrix = df[feature_cols].astype('float32').values
    cooccur = feature_matrix.T @ feature_matrix

    if distance_method == 'jaccard':
        # Jaccard distance - RECOMMENDED for binary data
        distance_matrix = squareform(pdist(feature_matrix.T, metric='jaccard'))
        distance_matrix = np.nan_to_num(distance_matrix, nan=1.0)
        print("  Using Jaccard distance (best for binary co-occurrence)")

    elif distance_method == 'cosine':
        # Cosine distance
        distance_matrix = squareform(pdist(feature_matrix.T, metric='cosine'))
        distance_matrix = np.nan_to_num(distance_matrix, nan=1.0)
        print("  Using Cosine distance")

    elif distance_method == 'correlation':
        # Correlation-based distance
        corr = np.corrcoef(feature_matrix.T)
        corr = np.nan_to_num(corr, nan=0.0)
        distance_matrix = np.sqrt(np.maximum(1 - corr, 0))
        print("  Using Correlation-based distance")

    elif distance_method == 'normalized_cooccur':
        # Normalized co-occurrence (FIXED version)
        cooccur_norm = cooccur / cooccur.max()
        distance_matrix = np.sqrt(np.maximum(1 - cooccur_norm, 0))
        print("  Using Normalized co-occurrence distance")

    elif distance_method == 'inverse_cooccur':
        # Inverse co-occurrence
        distance_matrix = 1.0 / (cooccur + 1.0)
        # Normalize to [0, 1]
        distance_matrix = distance_matrix / distance_matrix.max()
        print("  Using Inverse co-occurrence distance")

    elif distance_method == 'euclidean':
        distance_matrix = squareform(pdist(feature_matrix.T, metric='euclidean'))
        print("  Using Euclidean distance")

    elif distance_method == 'dice':
        distance_matrix = squareform(pdist(feature_matrix.T, metric='dice'))
        print("  Using Dice distance")

    elif distance_method == 'hamming':
        distance_matrix = squareform(pdist(feature_matrix.T, metric='hamming'))
        print("  Using Hamming distance")

    elif distance_method == 'yule':
        distance_matrix = squareform(pdist(feature_matrix.T, metric='yule'))
        print("  Using Yule distance")

    else:
        raise ValueError(f"Unknown distance method: {distance_method}")

    print(f"  Cooccurrence matrix range: [{cooccur.min():.4f}, {cooccur.max():.4f}]")
    print(f"  Distance matrix range: [{distance_matrix.min():.4f}, {distance_matrix.max():.4f}]")

    distance_matrix_df = pd.DataFrame(
        distance_matrix,
        index=feature_cols,
        columns=feature_cols
    )
    cooccur_matrix_df = pd.DataFrame(
        cooccur,
        index=feature_cols,
        columns=feature_cols
    )

    print(f"Distance matrix shape: {distance_matrix_df.shape}")

    return distance_matrix_df, cooccur_matrix_df

In [ ]:
def perform_mds(distance_matrix, n_components=2, random_state=42):
    """
    Perform Multi-Dimensional Scaling

    Parameters:
    -----------
    similarity_matrix : DataFrame or ndarray
        Similarity/co-occurrence matrix
    n_components : int
        Number of dimensions (default: 2)
    metric : bool
        Whether to use metric MDS (default: True)
    random_state : int
        Random seed

    Returns:
    --------
    mds_coords : ndarray
        MDS coordinates (n_features x n_components)
    stress : float
        MDS stress value
    """
    print(f"\nPerforming MDS (n_components={n_components}...)")

    distance_matrix = distance_matrix.values
    # Create MDS object

    mds = MDS(n_components=n_components,
              max_iter=300000,
              eps=1e-9,
              n_init=5,
              random_state=random_state,
              dissimilarity="precomputed",
              n_jobs=1)
    pos = mds.fit(distance_matrix).embedding_

    print(f"\n✓ MDS Complete!")
    print(f"  Stress: {mds.stress_:.4f}")

    nmds = MDS(n_components=n_components,
               metric_mds=False,
               max_iter=300000,
               eps=1e-12,
               dissimilarity="precomputed",
               random_state=random_state,
               n_jobs=1,
               n_init=1)
    npos = nmds.fit_transform(distance_matrix, init=pos)

    print(f"\n✓ NMDS Complete!")
    print(f"  Stress: {nmds.stress_:.4f}")

    #pos *= np.sqrt((feature_matrix.T ** 2).sum().sum()) / np.sqrt((pos ** 2).sum())
    #npos *= np.sqrt((feature_matrix.T ** 2).sum().sum()) / np.sqrt((npos ** 2).sum())

    #clf = PCA(n_components=2)

    #pos = clf.fit_transform(pos)
    #npos = clf.fit_transform(npos)

    # Fit MDS
    mds_coords = npos
    stress = nmds.stress_

    # Evaluate quality
    if stress < 0.05:
        quality = "⭐⭐⭐⭐⭐ Excellent"
    elif stress < 0.10:
        quality = "⭐⭐⭐⭐ Good"
    elif stress < 0.20:
        quality = "⭐⭐⭐ Fair"
    else:
        quality = "⭐⭐ Poor"

    print(f"\n✓ Entire process of MDS Complete!")
    print(f"  Stress: {stress:.4f}")
    print(f"  Quality: {quality}")

    return mds_coords, stress

In [ ]:
def create_mds_dataframe(mds_coords, feature_names, feature_types=None, n_components=2):
    """
    Create DataFrame from MDS coordinates

    Parameters:
    -----------
    mds_coords : ndarray
        MDS coordinates
    feature_names : list
        Feature/variable names
    feature_types : list, optional
        Types of features (e.g., 'platform', 'modality', 'task', 'model', 'topic')

    Returns:
    --------
    mds_df : DataFrame
        MDS coordinates with feature info
    """
    if n_components==2:
        mds_df = pd.DataFrame(
            mds_coords,
            columns=['MDS1', 'MDS2']
        )
    elif n_components==3:
        mds_df = pd.DataFrame(
            mds_coords,
            columns=['MDS1', 'MDS2', 'MDS3']
        )
    mds_df['feature'] = feature_names

    if feature_types is not None:
        mds_df['type'] = feature_types
    else:
        # Infer types from feature names
        def infer_type(name):
            name_lower = name.lower()
            if any(x in name_lower for x in ['_topic_']):
                return 'topic'
            elif any(x in name_lower for x in ['platform_']):
                return 'platform'
            elif any(x in name_lower for x in ['modality_']):
                return 'modality'
            elif any(x in name_lower for x in ['task_']):
                return 'task'
            elif any(x in name_lower for x in ['model_']):
                return 'model'
            else:
                return 'other'

        mds_df['type'] = mds_df['feature'].apply(infer_type)

    return mds_df

In [ ]:
def identify_topic_columns(df, topic_prefix='binary_topic_'):
    """
    Identify topic columns in the dataframe

    Parameters:
    -----------
    df : DataFrame
        Combined dataframe with features and topics
    topic_prefix : str
        Prefix for topic column names

    Returns:
    --------
    topic_cols : list
        List of topic column names
    feature_cols : list
        List of feature column names (non-topic, non-ID)
    """
    # Identify topic columns
    topic_cols = [col for col in df.columns if col.lower().startswith(topic_prefix) and not col.lower().endswith('_-1')]

    # Identify feature columns (excluding topics and ID columns)
    id_cols = ['filename', 'paper_id', 'id', 'Binary_Topic_-1', 'Prob_Topic_-1']
    feature_cols = [col for col in df.columns
                   if col not in topic_cols
                   and col not in id_cols
                   and col.lower() not in id_cols]

    print(f"Found {len(topic_cols)} topic columns: {topic_cols}")
    print(f"Found {len(feature_cols)} feature columns")

    return topic_cols, feature_cols

In [ ]:
from scipy.stats import pointbiserialr
def find_top_features_per_topic(df, topic_cols, feature_cols, n_top=10, threshold=None,
                                method='correlation'):
    """
    Find top N features most related to each topic

    Parameters:
    -----------
    df : DataFrame
        Combined dataframe
    topic_cols : list
        Topic column names
    feature_cols : list
        Feature column names
    n_top : int
        Number of top features to select per topic
    method : str
        'correlation' (for continuous topics) or 'cooccurrence' (for binary)

    Returns:
    --------
    topic_features : dict
        Dictionary mapping topic name to list of top features
    """
    print(f"\n{'='*70}")
    print(f"FINDING TOP {n_top} FEATURES PER TOPIC")
    print(f"Method: {method}")
    print(f"{'='*70}")

    topic_features = {}

    for topic in topic_cols:
        topic_values = df[topic].values

        # Calculate relationship with each feature
        relationships = {}

        for feature in feature_cols:
            feature_values = df[feature].values

            if method == 'correlation':
                # Point-biserial correlation (continuous vs binary)
                # Use this when topic is continuous (probability)
                if feature_values.sum() > 0:  # Skip if feature never appears
                    corr, pval = pointbiserialr(feature_values, topic_values)
                    relationships[feature] = abs(corr)  # Use absolute correlation
                else:
                    relationships[feature] = 0

            elif method == 'cooccurrence':
                # Simple co-occurrence weighted by topic strength
                # Sum of topic values where feature is present
                cooccur = (feature_values * topic_values).sum()
                relationships[feature] = cooccur

            elif method == 'mean_topic_value':
                # Mean topic value when feature is present
                if feature_values.sum() > 0:
                    mean_when_present = topic_values[feature_values == 1].mean()
                    mean_when_absent = topic_values[feature_values == 0].mean()
                    relationships[feature] = mean_when_present - mean_when_absent
                else:
                    relationships[feature] = 0

        # Sort and get top N
        sorted_features = sorted(relationships.items(), key=lambda x: x[1], reverse=True)
        top_features = [f for f, score in sorted_features[:n_top]]

        if threshold is not None:
            top_features = [f for f,score in sorted_features if score >= threshold]

        topic_features[topic] = {
            'features': top_features,
            'scores': {f: relationships[f] for f in top_features}
        }

        print(f"\n{topic}:")
        print(f"  Top {n_top} features:")
        for i, (feature, score) in enumerate(topic_features[topic]['scores'].items(),1):
            print(f"    {i}. {feature}: {score:.4f}")

    return topic_features

def create_filtered_dataset(df, topic_features, topic_cols, feature_cols):
    """
    Create filtered dataset with only top features per topic

    Parameters:
    -----------
    df : DataFrame
        Original dataframe
    topic_features : dict
        Dictionary from find_top_features_per_topic
    topic_cols : list
        Topic columns
    feature_cols : list
        All feature columns

    Returns:
    --------
    selected_features : list
        Unique features selected across all topics
    filtered_df : DataFrame
        Filtered dataframe with only selected features
    feature_topic_map : dict
        Maps each feature to topics it's associated with
    """
    print(f"\n{'='*70}")
    print("CREATING FILTERED DATASET")
    print(f"{'='*70}")

    # Collect all selected features (union across topics)
    all_selected = set()
    feature_topic_map = {}

    for topic, info in topic_features.items():
        for feature in info['features']:
            all_selected.add(feature)
            if feature not in feature_topic_map:
                feature_topic_map[feature] = []
            feature_topic_map[feature].append(topic)

    selected_features = sorted(list(all_selected))

    print(f"\nOriginal features: {len(feature_cols)}")
    print(f"Selected features: {len(selected_features)}")
    print(f"Reduction: {(1 - len(selected_features)/len(feature_cols))*100:.1f}%")

    # Show overlap statistics
    print(f"\nFeature overlap across topics:")
    overlap_counts = [len(topics) for topics in feature_topic_map.values()]
    for n_topics in range(1, max(overlap_counts)+1):
        count = sum(1 for c in overlap_counts if c == n_topics)
        if count > 0:
            print(f"  Features in {n_topics} topic(s): {count}")

    # Create filtered dataframe
    id_cols = [col for col in df.columns if col not in feature_cols and col not in topic_cols and not (col=='Binary_Topic_-1' or col=='Prob_Topic_-1')]
    filtered_df = df[id_cols + selected_features + topic_cols]

    return selected_features, filtered_df, feature_topic_map

In [ ]:
"""
3D MDS Visualization Functions
Interactive and static 3D visualizations for MDS results
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# Try to import plotly for interactive plots
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("⚠️ Plotly not available. Install with: pip install plotly")


def plot_3d_interactive(coords_3d, feature_names,
                        color_by='auto', sizes=None,
                        title='3D MDS Visualization',
                        save_html=None):
    """
    Create interactive 3D scatter plot using Plotly

    Parameters:
    -----------
    coords_3d : ndarray
        3D coordinates (n_features × 3)
    feature_names : list
        Feature names
    color_by : str, array, or dict
        How to color points:
        - 'auto': automatic coloring
        - array: numeric values for continuous coloring
        - dict: mapping feature_name -> category for discrete coloring
        - 'cluster': will perform clustering
    sizes : array, optional
        Size for each point
    title : str
        Plot title
    save_html : str, optional
        Path to save interactive HTML

    Returns:
    --------
    fig : plotly figure
    """
    if not PLOTLY_AVAILABLE:
        print("❌ Plotly not installed. Use plot_3d_static instead.")
        return None

    print("Creating interactive 3D plot...")

    # Prepare data
    df_plot = pd.DataFrame({
        'x': coords_3d[:, 0],
        'y': coords_3d[:, 1],
        'z': coords_3d[:, 2],
        'feature': feature_names
    })

    # Handle coloring
    if isinstance(color_by, dict):
        # Map features to categories
        df_plot['category'] = df_plot['feature'].map(color_by)
        df_plot['category'] = df_plot['category'].fillna('other')
        color_col = 'category'

    elif isinstance(color_by, (list, np.ndarray)):
        # Use provided values
        df_plot['color_value'] = color_by
        color_col = 'color_value'

    elif color_by == 'cluster':
        # Perform clustering
        from sklearn.cluster import KMeans
        n_clusters = min(10, len(feature_names) // 5)
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        clusters = kmeans.fit_predict(coords_3d)
        df_plot['cluster'] = [f'Cluster {i+1}' for i in clusters]
        color_col = 'cluster'

    else:
        # Auto: color by index or single color
        df_plot['index'] = range(len(feature_names))
        color_col = 'index'

    # Handle sizes
    if sizes is not None:
        df_plot['size'] = sizes
        size_col = 'size'
    else:
        df_plot['size'] = 10
        size_col = 'size'

    # Create figure
    fig = px.scatter_3d(
        df_plot,
        x='x', y='y', z='z',
        color=color_col,
        size=size_col,
        hover_name='feature',
        hover_data={'x': ':.3f', 'y': ':.3f', 'z': ':.3f'},
        title=title,
        labels={'x': 'MDS Dimension 1',
                'y': 'MDS Dimension 2',
                'z': 'MDS Dimension 3'}
    )

    # Update layout
    fig.update_traces(
        marker=dict(
            line=dict(width=1, color='DarkSlateGray'),
            opacity=0.8
        ),
        textposition='top center'
    )

    # Add text labels for all points
    for i, row in df_plot.iterrows():
        fig.add_trace(go.Scatter3d(
            x=[row['x']],
            y=[row['y']],
            z=[row['z']],
            mode='text',
            text=[row['feature']],
            textposition='top center',
            textfont=dict(size=8, color='black'),
            showlegend=False,
            hoverinfo='skip'
        ))

    fig.update_layout(
        scene=dict(
            xaxis=dict(title='MDS Dimension 1', backgroundcolor="rgb(230, 230,230)"),
            yaxis=dict(title='MDS Dimension 2', backgroundcolor="rgb(230, 230,230)"),
            zaxis=dict(title='MDS Dimension 3', backgroundcolor="rgb(230, 230,230)"),
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.5)
            )
        ),
        width=1200,
        height=800,
        title_font_size=16
    )

    # Save if requested
    if save_html:
        fig.write_html(save_html)
        print(f"✓ Interactive plot saved to {save_html}")

    print("✓ Interactive plot created (use fig.show() to display)")

    return fig

def plot_3d_projections(coords_3d, feature_names,
                        color_by='auto', sizes=None,
                        title='3D MDS: Pairwise Projections',
                        save_path=None):
    """
    Create 2D projections of all 3 pairwise combinations

    Parameters:
    -----------
    coords_3d : ndarray
        3D coordinates
    feature_names : list
        Feature names
    color_by : str, array, or dict
        Coloring scheme
    sizes : array, optional
        Point sizes
    title : str
        Overall title
    save_path : str, optional
        Path to save figure

    Returns:
    --------
    fig : matplotlib figure
    """
    print("Creating 2D projection plots...")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Handle coloring
    if isinstance(color_by, dict):
        categories = [color_by.get(f, 'other') for f in feature_names]
        unique_cats = sorted(set(categories))
        colors_palette = sns.color_palette("husl", len(unique_cats))
        color_map = dict(zip(unique_cats, colors_palette))
        colors = [color_map[cat] for cat in categories]
    elif isinstance(color_by, (list, np.ndarray)):
        colors = list(color_by)  # Convert to list
    else:
        colors = ['steelblue'] * len(feature_names)

    # Handle sizes
    if sizes is None:
        sizes = np.full(len(feature_names), 100)
    elif isinstance(sizes, (int, float)):
        sizes = np.full(len(feature_names), sizes)
    else:
        sizes = np.array(sizes)

    # Projections
    projections = [
        (0, 1, 'Dim 1 vs Dim 2'),
        (0, 2, 'Dim 1 vs Dim 3'),
        (1, 2, 'Dim 2 vs Dim 3')
    ]

    for ax, (dim1, dim2, proj_title) in zip(axes, projections):
        ax.scatter(coords_3d[:, dim1], coords_3d[:, dim2],
                  c=colors, s=sizes, alpha=0.7,
                  edgecolors='black', linewidth=1.5)

        # Add labels for subset
        n_labels = min(15, len(feature_names))
        if n_labels > 0:
            label_indices = np.random.choice(len(feature_names), n_labels, replace=False)
            for i in label_indices:
                ax.annotate(feature_names[i],
                           (coords_3d[i, dim1], coords_3d[i, dim2]),
                           xytext=(5, 5), textcoords='offset points',
                           fontsize=8, alpha=0.7)

        ax.set_xlabel(f'MDS Dimension {dim1+1}', fontsize=11, fontweight='bold')
        ax.set_ylabel(f'MDS Dimension {dim2+1}', fontsize=11, fontweight='bold')
        ax.set_title(proj_title, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='k', linestyle='--', alpha=0.2)
        ax.axvline(x=0, color='k', linestyle='--', alpha=0.2)

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✓ Projection plots saved to {save_path}")

    print("✓ Projection plots created")

    return fig


# Convenience function: create all visualizations at once
def visualize_3d_mds_complete(coords_3d, feature_names,
                             color_by='auto', sizes=None,
                             output_prefix='3d_mds'):
    """
    Create complete suite of 3D visualizations

    Parameters:
    -----------
    coords_3d : ndarray
        3D MDS coordinates
    feature_names : list
        Feature names
    color_by : str, array, or dict
        Coloring scheme
    sizes : array, optional
        Point sizes
    output_prefix : str
        Prefix for all output files

    Returns:
    --------
    figures : dict
        Dictionary of all created figures
    """
    print("\n" + "="*70)
    print("CREATING COMPLETE 3D VISUALIZATION SUITE")
    print("="*70)

    figures = {}

    # 1. Interactive plot
    if PLOTLY_AVAILABLE:
        print("\n1. Interactive 3D plot...")
        fig_interactive = plot_3d_interactive(
            coords_3d, feature_names,
            color_by=color_by, sizes=sizes,
            title='Interactive 3D MDS Visualization',
            save_html=f"{output_prefix}_interactive.html"
        )
        figures['interactive'] = fig_interactive

    # 5. Projections
    print("\n5. 2D projections...")
    fig_projections = plot_3d_projections(
        coords_3d, feature_names,
        color_by=color_by, sizes=sizes,
        title='3D MDS: Pairwise Projections',
        save_path=f"{output_prefix}_projections.png"
    )
    figures['projections'] = fig_projections

    print("\n" + "="*70)
    print("✓ COMPLETE! All visualizations created")
    print("\nGenerated files:")
    if PLOTLY_AVAILABLE:
        print(f"  • {output_prefix}_interactive.html (interactive 3D)")
    print(f"  • {output_prefix}_static.png (static 3D)")
    print(f"  • {output_prefix}_multiple_views.png (4 viewing angles)")
    print(f"  • {output_prefix}_clusters.png (with clustering)")
    if PLOTLY_AVAILABLE:
        print(f"  • {output_prefix}_clusters_interactive.html (clusters, interactive)")
    print(f"  • {output_prefix}_projections.png (2D projections)")
    print("="*70)

    return figures


In [ ]:
topic_cols, feature_cols = identify_topic_columns(combined_df, 'binary_topic_')
topic_features = find_top_features_per_topic(
        combined_df, topic_cols, feature_cols, n_top=10, method='mean_topic_value'
    )
selected_features, filtered_df, feature_topic_map = create_filtered_dataset(
        combined_df, topic_features, topic_cols, feature_cols
    )

In [ ]:
def plot_mds_scatter(mds_df, title='MDS: Terms and Topics in 2D Space',
                     figsize=(20, 20), save_path=None):
    """
    Create scatter plot of MDS results

    Parameters:
    -----------
    mds_df : DataFrame
        MDS coordinates with feature info
    title : str
        Plot title
    figsize : tuple
        Figure size
    save_path : str, optional
        Path to save figure
    """
    print("\nCreating MDS scatter plot...")

    fig, ax = plt.subplots(figsize=figsize)

    # Get unique types and colors
    types = mds_df['type'].unique()
    colors = sns.color_palette("husl", len(types))
    color_map = dict(zip(types, colors))

    # Plot each type
    for feat_type in types:
        subset = mds_df[mds_df['type'] == feat_type]
        cols = [col for col in subset.columns if col.startswith('MDS')]
        ax.scatter(
            subset[cols[0]],
            subset[cols[1]],
            c=[color_map[feat_type]],
            label=feat_type.capitalize(),
            s=150,
            alpha=0.7,
            edgecolors='black',
            linewidth=1.5
        )

        # Add labels
        for idx, row in subset.iterrows():
            ax.annotate(
                row['feature'],
                (row[cols[0]], row[cols[1]]),
                xytext=(5, 5),
                textcoords='offset points',
                fontsize=9,
                alpha=0.8
            )

    ax.set_xlabel('MDS Dimension 1', fontsize=12, fontweight='bold')
    ax.set_ylabel('MDS Dimension 2', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='best', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax.axvline(x=0, color='k', linestyle='--', alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to {save_path}")

    return fig


def plot_mds_heatmap(distance_matrix, feature_names, title='Feature Co-occurrence Heatmap',
                     figsize=(20, 20), save_path=None):
    """
    Create heatmap of similarity/co-occurrence matrix

    Parameters:
    -----------
    similarity_matrix : DataFrame or ndarray
        Similarity matrix
    feature_names : list
        Feature names
    title : str
        Plot title
    figsize : tuple
        Figure size
    save_path : str, optional
        Path to save figure
    """
    print("\nCreating co-occurrence heatmap...")

    fig, ax = plt.subplots(figsize=figsize)

    # Create heatmap
    sns.heatmap(
        distance_matrix,
        xticklabels=feature_names,
        yticklabels=feature_names,
        cmap='YlOrRd',
        cbar_kws={'label': 'Co-occurrence Count'},
        square=True,
        ax=ax,
        linewidths=0.5,
        linecolor='gray'
    )

    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Heatmap saved to {save_path}")

    return fig


def plot_dendrogram(distance_matrix, feature_names, title='Hierarchical Clustering Dendrogram',
                   figsize=(14, 8), save_path=None):
    """
    Create dendrogram from hierarchical clustering

    Parameters:
    -----------
    similarity_matrix : DataFrame or ndarray
        Similarity matrix
    feature_names : list
        Feature names
    title : str
        Plot title
    figsize : tuple
        Figure size
    save_path : str, optional
        Path to save figure
    """
    print("\nCreating dendrogram...")

    # Compute linkage
    condensed_dist = squareform(distance_matrix, checks=False)
    linkage_matrix = linkage(condensed_dist, method='average')

    # Create dendrogram
    fig, ax = plt.subplots(figsize=figsize)

    dendrogram(
        linkage_matrix,
        labels=feature_names,
        ax=ax,
        leaf_rotation=90,
        leaf_font_size=10,
        color_threshold=0.7*max(linkage_matrix[:, 2])
    )

    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Features', fontsize=12, fontweight='bold')
    ax.set_ylabel('Distance', fontsize=12, fontweight='bold')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Dendrogram saved to {save_path}")

    return fig


def export_results(mds_df, distance_matrix, similarity_matrix, output_prefix='mds_results'):
    """
    Export MDS results and matrices to CSV

    Parameters:
    -----------
    mds_df : DataFrame
        MDS coordinates
    similarity_matrix : DataFrame
        Similarity/co-occurrence matrix
    output_prefix : str
        Prefix for output filenames
    """
    print("\nExporting results...")

    # Export MDS coordinates
    mds_output = f"{output_prefix}_coordinates.csv"
    mds_df.to_csv(mds_output, index=False)
    print(f"MDS coordinates saved to {mds_output}")

    # Export distance matrix
    sim_output = f"{output_prefix}_distance_matrix.csv"
    distance_matrix.to_csv(sim_output)
    print(f"Distance matrix saved to {sim_output}")

    # Export similarity matrix
    sim_output = f"{output_prefix}_cooccurrence_matrix.csv"
    similarity_matrix.to_csv(sim_output)
    print(f"Co-occurrence matrix saved to {sim_output}")


def main_analysis(combined_df, paper_id_col='filename', reduce_to = 10, reduce_method = 'correlation', threshold=None,
                  output_prefix='outputs/method_theme/mds_analysis', distance_metric='euclidean', binary_topics=True,
                  create_plots=True, n_components=2):
    """
    Main analysis pipeline

    Parameters:
    -----------
    binary_csv_path : str
        Path to binary categories CSV
    topics_csv_path : str
        Path to topics CSV
    paper_id_col : str
        Column name for paper ID
    output_prefix : str
        Prefix for output files
    create_plots : bool
        Whether to create visualizations

    Returns:
    --------
    results : dict
        Dictionary containing all results
    """
    print("="*70)
    print("MDS ANALYSIS: Terms and Topics Co-occurrence")
    print("="*70)

    if binary_topics:
        topic_cols, feature_cols = identify_topic_columns(combined_df, 'binary_topic_')
    else:
        topic_cols, feature_cols = identify_topic_columns(combined_df, 'prob_topic_')
    topic_features = find_top_features_per_topic(
            combined_df, topic_cols, feature_cols, n_top=reduce_to, method=reduce_method, threshold=threshold
        )
    selected_features, filtered_df, feature_topic_map = create_filtered_dataset(
            combined_df, topic_features, topic_cols, feature_cols
        )

    # Identify feature columns (exclude paper_id if present)
    feature_cols = [col for col in filtered_df.columns if col != paper_id_col]
    print(f"Dataset: {len(combined_df)} papers, {len(feature_cols)} features")

    # Compute dissimilar matrix
    distance_matrix, cooccur_matrix = get_distance_matrix(filtered_df, feature_cols, distance_method=distance_metric)

    # Perform MDS
    mds_coords, stress = perform_mds(distance_matrix, n_components=n_components)

    # Create MDS DataFrame
    mds_df = create_mds_dataframe(mds_coords, feature_cols, n_components=n_components)

    # Export results
    export_results(mds_df, distance_matrix, cooccur_matrix, output_prefix)

    # Create visualizations
    if create_plots and n_components == 2:
        plot_mds_scatter(
            mds_df,
            title='MDS: Terms and Topics in 2D Space (Co-occurrence Based)',
            save_path=f"{output_prefix}_scatter.png"
        )

        plot_mds_heatmap(
            distance_matrix,
            feature_cols,
            title='Feature Co-occurrence Matrix',
            save_path=f"{output_prefix}_heatmap.png"
        )

        plot_dendrogram(
            distance_matrix,
            feature_cols,
            title='Hierarchical Clustering of Features',
            save_path=f"{output_prefix}_dendrogram.png"
        )

        plt.show()

    if create_plots and n_components == 3:
        visualize_3d_mds_complete(
        coords_3d=np.array(mds_df.values),  # shape: (n_features, 3)
        feature_names=feature_cols,
        output_prefix='my_3d_mds'
    )
        for col1 in ['MDS1', 'MDS2']:
            for col2 in ['MDS2', 'MDS3']:
                if not col1==col2:
                    plot_mds_scatter(
                                    mds_df[[col1, col2, 'type', 'feature']],
                                    title='MDS: Terms and Topics in 2D Space (Co-occurrence Based)',
                                    save_path=f"{output_prefix}_scatter.png"
                                )

        plt.show()


    # Compile results
    results = {
        'mds_coordinates': mds_df,
        'distance_matrix': distance_matrix,
        'stress': stress,
        'n_papers': len(combined_df),
        'n_features': len(feature_cols)
    }

    print("\n" + "="*70)
    print("Analysis completed successfully!")
    print(f"Number of papers: {results['n_papers']}")
    print(f"Number of features: {results['n_features']}")
    print(f"MDS Stress: {results['stress']:.4f}")
    print("="*70)

    return results

In [ ]:
filtered_df

In [ ]:
binary_df, topics_df = load_data('data/term_mapping/Gemini_2_flash_all/dataset_with_binary_columns.csv', 'data/term_mapping/Gemini_2_5_pro/dataset_with_binary_columns.csv', 'data/topic_modeling/probability_distribution.csv','data/topic_modeling/meta_data.csv', binary_topics=False)

In [ ]:
binary_df

In [ ]:
topics_df

In [ ]:
combined_df = combine_datasets(binary_df, topics_df, 'filename')
combined_df

In [ ]:
combined_df.to_csv('data/term_mapping/Binary_encodings.csv')

In [ ]:
results = main_analysis(
            combined_df,
            paper_id_col='filename',  # Adjust if your ID column has a different name
            output_prefix='outputs/method_theme/mds_analysis',
            distance_metric = 'cosine',
            n_components = 2,
            reduce_to=5,
            threshold=0.15,
            reduce_method= 'correlation',
            create_plots=True,
            binary_topics=False,
    )

In [ ]:
for method in ['jaccard', 'cosine', 'hamming', 'yule', 'dice']:
    for dim in [5, 10, 15]:
        results = main_analysis(
            combined_df,
            paper_id_col='filename',  # Adjust if your ID column has a different name
            output_prefix='mds_analysis',
            distance_metric = 'yule',
            n_components = 2,
            reduce_to=dim,
            reduce_method= 'correlation',
            create_plots=False
    )

In [ ]:
results = main_analysis(
        binary_df,
        topics_df,
        paper_id_col='filename',  # Adjust if your ID column has a different name
        output_prefix='mds_analysis',
        distance_metric = 'cosine',
        create_plots=False
    )

In [ ]:
results['mds_coordinates']

In [ ]:
from umap import UMAP

In [ ]:
# For 2D visualization
umap_2d = UMAP(
    n_components=2,
    metric='precomputed',
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)
coords_2d = umap_2d.fit_transform(distance_matrix)

In [ ]:
def main_analysis(combined_df, paper_id_col='filename', reduce_to = 10, reduce_method = 'correlation', threshold=None,
                  output_prefix='outputs/method_theme/mds_analysis', distance_metric='euclidean', reduction='MDS',
                  create_plots=True, n_components=2):
    """
    Main analysis pipeline

    Parameters:
    -----------
    binary_csv_path : str
        Path to binary categories CSV
    topics_csv_path : str
        Path to topics CSV
    paper_id_col : str
        Column name for paper ID
    output_prefix : str
        Prefix for output files
    create_plots : bool
        Whether to create visualizations

    Returns:
    --------
    results : dict
        Dictionary containing all results
    """
    print("="*70)
    print("MDS ANALYSIS: Terms and Topics Co-occurrence")
    print("="*70)

    topic_cols, feature_cols = identify_topic_columns(combined_df, 'binary_topic_')
    topic_features = find_top_features_per_topic(
            combined_df, topic_cols, feature_cols, n_top=reduce_to, method=reduce_method, threshold=threshold
        )
    selected_features, filtered_df, feature_topic_map = create_filtered_dataset(
            combined_df, topic_features, topic_cols, feature_cols
        )

    # Identify feature columns (exclude paper_id if present)
    feature_cols = [col for col in filtered_df.columns if col != paper_id_col]
    print(f"Dataset: {len(combined_df)} papers, {len(feature_cols)} features")

    # Compute dissimilar matrix
    distance_matrix = get_distance_matrix(filtered_df, feature_cols, distance_method=distance_metric)

    if reduction == 'UMAP':
        umap_2d = UMAP(
                    n_components=2,
                    metric='precomputed',
                    n_neighbors=15,
                    min_dist=0.1,
                    random_state=42
                    )
        mds_coords = umap_2d.fit_transform(distance_matrix)

    elif reduction == 'MDS':
        # Perform MDS
        mds_coords, stress = perform_mds(distance_matrix, n_components=n_components)

    # Create MDS DataFrame
    mds_df = create_mds_dataframe(mds_coords, feature_cols, n_components=n_components)

    # Export results
    export_results(mds_df, distance_matrix, output_prefix)

    # Create visualizations
    if create_plots and n_components == 2:
        plot_mds_scatter(
            mds_df,
            title='MDS: Terms and Topics in 2D Space (Co-occurrence Based)',
            save_path=f"{output_prefix}_scatter.png"
        )

        plot_mds_heatmap(
            distance_matrix,
            feature_cols,
            title='Feature Co-occurrence Matrix',
            save_path=f"{output_prefix}_heatmap.png"
        )

        plot_dendrogram(
            distance_matrix,
            feature_cols,
            title='Hierarchical Clustering of Features',
            save_path=f"{output_prefix}_dendrogram.png"
        )

        plt.show()

    if create_plots and n_components == 3:
        visualize_3d_mds_complete(
        coords_3d=np.array(mds_df.values),  # shape: (n_features, 3)
        feature_names=feature_cols,
        output_prefix='my_3d_mds'
    )
        for col1 in ['MDS1', 'MDS2']:
            for col2 in ['MDS2', 'MDS3']:
                if not col1==col2:
                    plot_mds_scatter(
                                    mds_df[[col1, col2, 'type', 'feature']],
                                    title='MDS: Terms and Topics in 2D Space (Co-occurrence Based)',
                                    save_path=f"{output_prefix}_scatter.png"
                                )

        plt.show()


    # Compile results
    results = {
        'mds_coordinates': mds_df,
        'distance_matrix': distance_matrix,
        'stress': stress,
        'n_papers': len(combined_df),
        'n_features': len(feature_cols)
    }

    print("\n" + "="*70)
    print("Analysis completed successfully!")
    print(f"Number of papers: {results['n_papers']}")
    print(f"Number of features: {results['n_features']}")
    print(f"MDS Stress: {results['stress']:.4f}")
    print("="*70)

    return results

In [ ]:
results = main_analysis(
            combined_df,
            paper_id_col='filename',  # Adjust if your ID column has a different name
            output_prefix='mds_analysis',
            distance_metric = 'dice',
            n_components = 2,
            reduce_to=5,
            threshold=0.2,
            reduce_method= 'correlation',
            create_plots=True,
            reduction='UMAP',
    )